In [ ]:
import os
import pandas as pd
import numpy as np
import random
import logging
import re

def limpar_mapeamento_e_isolar(caminho_arquivo):
    # 1. Carregar usando pandas ignorando o cabeçalho real
    # sep='\t' porque o USPTO usa tabs. 
    # usecols=[0] para pegar apenas a coluna ReactionSmiles
    df = pd.read_csv(caminho_arquivo, sep='\t', low_memory=False)
    
    # Se a primeira coluna não se chamar 'ReactionSmiles', 
    # podemos forçar a captura da primeira coluna independente do nome
    coluna_reacao = df.iloc[:, 0].astype(str)
    
    lista_limpa = []
    
    for smi in coluna_reacao:
        # Pular linhas vazias ou cabeçalhos acidentais
        if smi.startswith('ReactionSmiles') or len(smi) < 1:
            continue
            
        # A limpeza principal:
        # [CH2:2] vira [CH2]
        smi_limpo = re.sub(r':\d+', '', smi)
        
        # Opcional: remover colchetes desnecessários que o mapeamento deixa
        # [C] vira C, mas [CH2] continua [CH2]. RDKit faz isso melhor, 
        # mas via Regex ajuda o LLM a ter menos tokens.
        
        lista_limpa.append(smi_limpo)
        
    return lista_limpa

# Exemplo com a sua string:
exemplo = "[Br:1][CH2:2][CH2:3][OH:4].[CH2:5]([S:7](Cl)(=[O:9])=[O:8])[CH3:6].CCOCC>C(N(CC)CC)C>[CH2:5]([S:7]([O:4][CH2:3][CH2:2][Br:1])(=[O:9])=[O:8])[CH3:6]"
resultado = re.sub(r':\d+', '', exemplo)
print(f"Antes: {exemplo}")
print(f"Depois: {resultado}")

df1 = limpar_mapeamento_e_isolar("1976_Sep2016_USPTOgrants_smiles.rsmi")
print(len(df1))
df2 = limpar_mapeamento_e_isolar("2001_Sep2016_USPTOapplications_smiles.rsmi")
print(len(df2))

Antes: [Br:1][CH2:2][CH2:3][OH:4].[CH2:5]([S:7](Cl)(=[O:9])=[O:8])[CH3:6].CCOCC>C(N(CC)CC)C>[CH2:5]([S:7]([O:4][CH2:3][CH2:2][Br:1])(=[O:9])=[O:8])[CH3:6]
Depois: [Br][CH2][CH2][OH].[CH2]([S](Cl)(=[O])=[O])[CH3].CCOCC>C(N(CC)CC)C>[CH2]([S]([O][CH2][CH2][Br])(=[O])=[O])[CH3]
1808937
1939253


In [5]:
# 1. Juntar tudo
all_raw = df1 + df2

# 2. Remover duplicatas exatas
unique_smiles = list(set(all_raw))
print(f"Únicos: {len(unique_smiles)}")

# 3. Limpeza Fina
dataset_final = []
for s in unique_smiles:
    # Apenas reações que tenham reagente e produto
    if ">" not in s: continue 
    
    # Limite de tamanho para o Llama 3 não se perder
    if len(s) > 450: continue
    
    # (Opcional mas recomendado) Validar com RDKit
    # if not validar_reacao(s): continue
    
    dataset_final.append(s)

print(f"Dataset pronto para treino: {len(dataset_final)}")

Únicos: 1484184
Dataset pronto para treino: 1366183


In [ ]:
dataset_final

In [ ]:
from rdkit import Chem
import pandas as pd

from rdkit import rdBase
rdBase.DisableLog('rdApp.*') # Silencia os erros de parse no console

def limpar_componente(smiles_componente):
    if not smiles_componente or pd.isna(smiles_componente):
        return ""
    
    # Remove qualquer metadado que venha após um espaço ou barra vertical comum no USPTO
    # Ex: "CCO |1,2,3|" -> "CCO"
    smiles_componente = smiles_componente.split('|')[0].split(' ')[0]
    
    moleculas = smiles_componente.split('.')
    moleculas_limpas = []
    
    for mol_smi in moleculas:
        # PRÉ-FILTRO: Se a string for só números ou muito curta, ignore antes de chamar o RDKit
        if not mol_smi or mol_smi.isdigit() or len(mol_smi) < 1:
            continue
            
        mol = Chem.MolFromSmiles(mol_smi)
        if mol:
            moleculas_limpas.append(Chem.MolToSmiles(mol, canonical=True))
    
    return ".".join(moleculas_limpas)

def higienizar_reacao_completa(reacao_smi):
    if not reacao_smi or ">" not in reacao_smi:
        return None
        
    partes = reacao_smi.split('>')
    
    # O split de 'R>>P' ou 'R>A>P' sempre resultará em 3 partes
    if len(partes) == 3:
        r_bruto, a_bruto, p_bruto = partes[0], partes[1], partes[2]
        
        reagentes = limpar_componente(r_bruto)
        produtos = limpar_componente(p_bruto)
        
        # Agentes podem ser vazios, então limpamos apenas se houver algo
        agentes = limpar_componente(a_bruto) if a_bruto.strip() else ""
        
        # REGRA DE OURO: Para ser uma reação, precisa ter reagente e produto válidos
        if reagentes and produtos:
            # Remonta com as setas. Se agentes for vazio, vira "R>>P"
            return f"{reagentes}>{agentes}>{produtos}"
            
    return None # Se cair aqui, o dropna exclui (ex: reação sem produto)

# 1. Transformar sua lista em DataFrame se ainda não fez
df = pd.DataFrame(dataset_final, columns=['smiles_bruto'])

# 2. Aplicar a higienização (isso pode demorar uns minutos para 1.3M)
print("Iniciando higienização profunda...")
df['smiles_higienizado'] = df['smiles_bruto'].apply(higienizar_reacao_completa)

# 3. Remover as reações que retornaram None (inválidas)
df_final = df.dropna(subset=['smiles_higienizado']).drop_duplicates(subset=['smiles_higienizado'])

print(f"Dataset Final Higienizado: {len(df_final)}")

Iniciando higienização profunda...
Dataset Final Higienizado: 1351092


In [16]:
df_final["smiles_higienizado"].to_csv("dados_limpos.csv", index=False)
df_final["smiles_higienizado"]

0          N#C[S-].[NH4+].II.Cc1c(N)cccc1Cl.O>CO>Cc1c(N)c...
1          COc1ccc(CO)cc1Oc1cccc(Cl)c1.c1ccc(P(c2ccccc2)c...
2          O=C(O)OCc1ccccc1.O=C(Cl)C(=O)Cl>ClCCl.CN(C)C=O...
3          COc1nc(S(C)(=O)=O)nc2oc(-c3ccc(C4(NC(=O)OC(C)(...
4          CNC(=O)c1cccc(N)c1.O=C(c1cccc2ccccc12)c1c[nH]c...
                                 ...                        
1366177    COC(=O)C(F)(F)C(F)(F)c1n[nH]c(-c2ccc([N+](=O)[...
1366178    Nc1cccc(Cl)c1[N+](=O)[O-].CC(C)(C)OC(=O)N1CCNC...
1366180    CCc1cn([C@H]2C[C@@](O)(C(C)=O)[C@@H](CO)O2)c(=...
1366181    COC(=O)c1cc(C2=NOC(c3ccccc3)C2)nc(C)n1.[OH-].[...
1366182    COC(=O)C1=Cc2cc(Cl)ccc2OCC1.O=[N+]([O-])O>O>CO...
Name: smiles_higienizado, Length: 1351092, dtype: object

In [23]:
reacoes_total = pd.read_csv("dados_limpos.csv")
reacoes_total

,smiles_higienizado
0,N#C[S-].[NH4+].II.Cc1c(N)cccc1Cl.O>CO>Cc1c(N)c...
1,COc1ccc(CO)cc1Oc1cccc(Cl)c1.c1ccc(P(c2ccccc2)c...
2,O=C(O)OCc1ccccc1.O=C(Cl)C(=O)Cl>ClCCl.CN(C)C=O...
3,COc1nc(S(C)(=O)=O)nc2oc(-c3ccc(C4(NC(=O)OC(C)(...
4,CNC(=O)c1cccc(N)c1.O=C(c1cccc2ccccc12)c1c[nH]c...
...,...
1351087,COC(=O)C(F)(F)C(F)(F)c1n[nH]c(-c2ccc([N+](=O)[...
1351088,Nc1cccc(Cl)c1[N+](=O)[O-].CC(C)(C)OC(=O)N1CCNC...
1351089,CCc1cn([C@H]2C[C@@](O)(C(C)=O)[C@@H](CO)O2)c(=...
1351090,COC(=O)c1cc(C2=NOC(c3ccccc3)C2)nc(C)n1.[OH-].[...


In [24]:
def separar_input_output(smi):
    # O split em '>' sempre gera 3 partes para R>A>P ou R>>P
    partes = smi.split('>')
    
    if len(partes) == 3:
        reagentes = partes[0]
        agentes = partes[1]
        produtos = partes[2]
        
        # Se agentes não for vazio, junta com ponto
        if agentes.strip():
            # Garante que não haja pontos duplicados no encontro
            input_context = f"{reagentes.strip('.')}.{agentes.strip('.')}"
        else:
            input_context = reagentes
            
        return input_context, produtos
    
    return None, None

# Aplicando a função
print("Separando colunas de Reagentes (Input) e Produtos (Output)...")

reacoes = pd.DataFrame()
# O zip(*) descompacta a tupla retornada pela função em duas colunas
reacoes['input_reagentes'], reacoes['output_produtos'] = zip(*reacoes_total['smiles_higienizado'].apply(separar_input_output))

# Remover possíveis falhas (embora o dataset já esteja higienizado)
reacoes = reacoes.dropna(subset=['input_reagentes', 'output_produtos'])

# Exibir o resultado
print(f"Total de reações prontas: {len(reacoes)}")
reacoes

Separando colunas de Reagentes (Input) e Produtos (Output)...
Total de reações prontas: 1351086


,input_reagentes,output_produtos
0,N#C[S-].[NH4+].II.Cc1c(N)cccc1Cl.O.CO,Cc1c(N)ccc(SC#N)c1Cl
1,COc1ccc(CO)cc1Oc1cccc(Cl)c1.c1ccc(P(c2ccccc2)c...,COc1ccc(CBr)cc1Oc1cccc(Cl)c1
2,O=C(O)OCc1ccccc1.O=C(Cl)C(=O)Cl.ClCCl.CN(C)C=O,O=C(Cl)COCc1ccccc1
3,COc1nc(S(C)(=O)=O)nc2oc(-c3ccc(C4(NC(=O)OC(C)(...,CNc1nc(OC)c2c(-c3ccccc3)c(-c3ccc(C4(NC(=O)OC(C...
4,CNC(=O)c1cccc(N)c1.O=C(c1cccc2ccccc12)c1c[nH]c...,CNC(=O)c1cccc(Nc2ncnc3[nH]cc(C(=O)c4cccc5ccccc...
...,...,...
1351087,COC(=O)C(F)(F)C(F)(F)c1n[nH]c(-c2ccc([N+](=O)[...,COC(=O)C(F)(F)C(F)(F)c1n[nH]c(-c2ccc(N)cc2)n1
1351088,Nc1cccc(Cl)c1[N+](=O)[O-].CC(C)(C)OC(=O)N1CCNC...,CC(C)(C)OC(=O)N1CCN(c2cccc(N)c2[N+](=O)[O-])CC1
1351089,CCc1cn([C@H]2C[C@@](O)(C(C)=O)[C@@H](CO)O2)c(=...,CCc1cn([C@H]2C[C@H](O)[C@@H](COP(=O)(O)O)O2)c(...
1351090,COC(=O)c1cc(C2=NOC(c3ccccc3)C2)nc(C)n1.[OH-].[...,Cc1nc(C(=O)O)cc(C2=NOC(c3ccccc3)C2)n1


In [28]:
# Calcular o comprimento das strings
reacoes['len_input'] = reacoes['input_reagentes'].str.len()
reacoes['len_output'] = reacoes['output_produtos'].str.len()

print("Estatísticas de comprimento (em caracteres):")
print(reacoes[['len_input', 'len_output']].describe(percentiles=[0.9, 0.95, 0.99]))

reacoes[['input_reagentes', 'output_produtos']].to_csv("dados_reacoes.csv", index=False)

Estatísticas de comprimento (em caracteres):
          len_input    len_output
count  1.351086e+06  1.351086e+06
mean   7.649478e+01  4.248889e+01
std    3.182627e+01  1.618980e+01
min    3.000000e+00  1.000000e+00
50%    7.100000e+01  4.100000e+01
90%    1.160000e+02  6.400000e+01
95%    1.340000e+02  7.000000e+01
99%    1.890000e+02  8.300000e+01
max    3.590000e+02  2.690000e+02
